# Tunix-Med: Baseline Medical Knowledge Evaluation

This notebook measures how well the **pre-trained `google/gemma-3-1b-it` model** answers a set of 25 medical questions based on cardiology clinical guidelines **before any fine-tuning**. The scores produced here become the **baseline** we compare against after domain-specific training with **Tunix**.

### Evaluation pipeline

For each question the notebook:
1. Generates an answer using the pre-trained model.
2. Computes **perplexity** of the reference answer.
3. Scores the answer with three independent methods:
   - **Keyword matching**
   - **Semantic similarity** (Sentence-BERT)
   - **AI judge** (LLM-as-a-Judge)
4. Aggregates results by question difficulty.

In [1]:
import os
import gc
import warnings
import torch
import pandas as pd
import re
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
from sentence_transformers import SentenceTransformer, util


def init():
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    token = os.environ.get("HF_TOKEN")
    if token:
        login(token=token)
    torch.cuda.empty_cache()
    gc.collect()
    warnings.filterwarnings("ignore")


def info_device():
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print(f"Using device: {device}")
    return device


def is_bfloat16_supported():
    if torch.backends.mps.is_available():
        return True
    return torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8


init()
device = info_device()
dtype = torch.bfloat16 if is_bfloat16_supported() else torch.float16
print(f"Using dtype: {dtype}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Using device: cuda
Using dtype: torch.bfloat16


## 1. Medical Evaluation Dataset
We use a curated set of 25 questions based on cardiology clinical guidelines (e.g., ESC/ACC/AHA). These questions range from easy (first-line treatments) to hard (complex mechanisms or specific thresholds).

In [2]:
medical_qa = [
    # --- HYPERTENSION (3) ---
    {
        "Question": "What is the recommended first-line treatment for uncomplicated hypertension according to ESC guidelines?",
        "Answer": "ACE inhibitors or ARBs, combined with a calcium channel blocker or a thiazide/thiazide-like diuretic.",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Hypertension",
    },
    {
        "Question": "What is the definition of Stage 1 hypertension according to the 2017 ACC/AHA guidelines?",
        "Answer": "Systolic blood pressure 130–139 mmHg or diastolic blood pressure 80–89 mmHg.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Hypertension",
    },
    {
        "Question": "What is the definition of hypertensive emergency, and what is the initial target for blood pressure reduction in the first hour?",
        "Answer": "Hypertensive emergency is severe hypertension (typically SBP > 180 mmHg) with acute end-organ damage. BP should be reduced by no more than 25% in the first hour using IV agents (e.g., labetalol, nicardipine).",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Hypertension",
    },
    # --- HEART FAILURE (3) ---
    {
        "Question": "In patients with heart failure with reduced ejection fraction (HFrEF), what are the 'four pillars' of medical therapy?",
        "Answer": "ACEi/ARNI, beta-blockers, MRA (mineralocorticoid receptor antagonists), and SGLT2 inhibitors.",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Heart Failure",
    },
    {
        "Question": "What is the ejection fraction threshold that defines heart failure with preserved ejection fraction (HFpEF) according to ESC 2021 guidelines?",
        "Answer": "LVEF ≥ 50%, in the presence of relevant structural/functional cardiac abnormalities and elevated natriuretic peptides.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Heart Failure",
    },
    {
        "Question": "What is the mechanism of action of sacubitril in the ARNI combination (sacubitril/valsartan)?",
        "Answer": "Sacubitril inhibits neprilysin, preventing degradation of natriuretic peptides (ANP, BNP), leading to vasodilation, natriuresis, and reduced cardiac remodelling.",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Heart Failure",
    },
    # --- DYSLIPIDAEMIA / PREVENTION (3) ---
    {
        "Question": "What is the target LDL-C level for patients at very high cardiovascular risk according to the 2019 ESC/EAS guidelines?",
        "Answer": "< 1.4 mmol/L (55 mg/dL) and a reduction of ≥ 50% from baseline.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Dyslipidaemia",
    },
    {
        "Question": "What is the first-line pharmacological treatment for hypercholesterolaemia?",
        "Answer": "High-intensity statin therapy (e.g., atorvastatin 40–80 mg or rosuvastatin 20–40 mg daily).",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Dyslipidaemia",
    },
    {
        "Question": "What is the mechanism of action of PCSK9 inhibitors, and when are they indicated?",
        "Answer": "PCSK9 inhibitors (e.g., evolocumab, alirocumab) block the PCSK9 protein, preventing LDL receptor degradation and thus increasing LDL clearance. They are indicated in very high-risk patients not achieving LDL targets on maximum tolerated statin ± ezetimibe.",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Dyslipidaemia",
    },
    # --- ACUTE CORONARY SYNDROME (3) ---
    {
        "Question": "In the context of acute coronary syndrome, what does the 'TIMI score' predict?",
        "Answer": "The risk of death and ischemic events in patients with UA/NSTEMI or STEMI.",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Acute Coronary Syndrome",
    },
    {
        "Question": "What is the door-to-balloon time target for primary PCI in STEMI according to ESC guidelines?",
        "Answer": "< 60 minutes from first medical contact if the patient presents directly to a PCI-capable centre.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Acute Coronary Syndrome",
    },
    {
        "Question": "What dual antiplatelet therapy (DAPT) regimen is recommended after primary PCI for STEMI?",
        "Answer": "Aspirin plus a potent P2Y12 inhibitor (ticagrelor or prasugrel preferred over clopidogrel) for 12 months, unless there is a high bleeding risk.",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Acute Coronary Syndrome",
    },
    # --- ATRIAL FIBRILLATION / ARRHYTHMIAS (4) ---
    {
        "Question": "What is the first-line pharmacological treatment for rate control in atrial fibrillation?",
        "Answer": "Beta-blockers or non-dihydropyridine calcium channel blockers (diltiazem or verapamil).",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Arrhythmias",
    },
    {
        "Question": "According to ESC guidelines, what is the recommended anticoagulation therapy for non-valvular atrial fibrillation in eligible patients?",
        "Answer": "Direct oral anticoagulants (DOACs) are preferred over vitamin K antagonists (VKAs).",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Arrhythmias",
    },
    {
        "Question": "What is the CHA₂DS₂-VASc score used for, and at what threshold is anticoagulation recommended in males?",
        "Answer": "It estimates stroke risk in atrial fibrillation; anticoagulation is recommended at a score ≥ 2 in males (≥ 3 in females).",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Arrhythmias",
    },
    {
        "Question": "What ECG finding is pathognomonic of Wolff-Parkinson-White (WPW) syndrome?",
        "Answer": "A short PR interval (< 120 ms), a delta wave (slurred QRS upstroke), and a wide QRS complex.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Arrhythmias",
    },
    # --- VALVULAR HEART DISEASE (3) ---
    {
        "Question": "What are the classic auscultatory findings of severe aortic stenosis?",
        "Answer": "A harsh crescendo-decrescendo systolic ejection murmur at the right upper sternal border, radiating to the carotids, with a slow-rising and low-volume pulse (pulsus parvus et tardus).",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Valvular Heart Disease",
    },
    {
        "Question": "According to ESC guidelines, what is the threshold for surgical aortic valve replacement (SAVR) or TAVI in symptomatic severe aortic stenosis?",
        "Answer": "AVA < 1.0 cm², mean gradient > 40 mmHg, or peak velocity > 4 m/s with symptoms (angina, syncope, or dyspnoea). Intervention is indicated (Class I) once symptoms are present.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Valvular Heart Disease",
    },
    {
        "Question": "What is the underlying mechanism of functional (secondary) mitral regurgitation in dilated cardiomyopathy?",
        "Answer": "Left ventricular dilation causes papillary muscle displacement and annular dilation, leading to incomplete mitral leaflet coaptation without intrinsic leaflet pathology.",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Valvular Heart Disease",
    },
    # --- CARDIOMYOPATHIES (3) ---
    {
        "Question": "What is the most common genetic cause of hypertrophic cardiomyopathy (HCM)?",
        "Answer": "Mutations in genes encoding sarcomeric proteins, most commonly MYH7 (beta-myosin heavy chain) and MYBPC3 (myosin-binding protein C), inherited in an autosomal dominant pattern.",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Cardiomyopathy",
    },
    {
        "Question": "What physical manoeuvre increases the murmur of hypertrophic obstructive cardiomyopathy (HOCM), and why?",
        "Answer": "The Valsalva manoeuvre (strain phase) increases the murmur by reducing preload, which decreases LV volume and worsens dynamic outflow tract obstruction.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Cardiomyopathy",
    },
    {
        "Question": "What is the first-line treatment for acute myocarditis with haemodynamic compromise?",
        "Answer": "Supportive care including diuretics, ACE inhibitors, and beta-blockers as tolerated; mechanical circulatory support (IABP or VA-ECMO) in refractory cardiogenic shock. Immunosuppression may be considered in biopsy-proven giant cell or autoimmune myocarditis.",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Cardiomyopathy",
    },
    # --- PERICARDIAL DISEASE (2) ---
    {
        "Question": "What is the first-line treatment for acute pericarditis?",
        "Answer": "NSAIDs (e.g., ibuprofen or aspirin) plus colchicine for 3 months to reduce recurrence risk.",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Pericardial Disease",
    },
    {
        "Question": "What are Beck's triad findings in cardiac tamponade?",
        "Answer": "Hypotension, elevated jugular venous pressure (JVP), and muffled heart sounds.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Pericardial Disease",
    },
    # --- AORTIC DISEASE (1) ---
    {
        "Question": "According to ESC guidelines, what is the diameter threshold for elective surgical repair of an ascending aortic aneurysm in a non-syndromic patient?",
        "Answer": "≥ 55 mm in diameter; the threshold is lower (≥ 50 mm) in patients with bicuspid aortic valve or connective tissue disorders such as Marfan syndrome.",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Aortic Disease",
    },
]

data = pd.DataFrame(medical_qa)
print(f"Total questions: {len(data)}")
print(data.groupby(["Sub-topic", "Difficulty"]).size().to_string())
data

Total questions: 25
Sub-topic                Difficulty
Acute Coronary Syndrome  Easy          1
                         Hard          1
                         Medium        1
Aortic Disease           Hard          1
Arrhythmias              Easy          2
                         Medium        2
Cardiomyopathy           Hard          2
                         Medium        1
Dyslipidaemia            Easy          1
                         Hard          1
                         Medium        1
Heart Failure            Easy          1
                         Hard          1
                         Medium        1
Hypertension             Easy          1
                         Hard          1
                         Medium        1
Pericardial Disease      Easy          1
                         Medium        1
Valvular Heart Disease   Easy          1
                         Hard          1
                         Medium        1


,Question,Answer,Difficulty,Category,Sub-topic
0,What is the recommended first-line treatment f...,"ACE inhibitors or ARBs, combined with a calciu...",Easy,Cardiology,Hypertension
1,What is the definition of Stage 1 hypertension...,Systolic blood pressure 130–139 mmHg or diasto...,Medium,Cardiology,Hypertension
2,What is the definition of hypertensive emergen...,Hypertensive emergency is severe hypertension ...,Hard,Cardiology,Hypertension
3,In patients with heart failure with reduced ej...,"ACEi/ARNI, beta-blockers, MRA (mineralocortico...",Easy,Cardiology,Heart Failure
4,What is the ejection fraction threshold that d...,"LVEF ≥ 50%, in the presence of relevant struct...",Medium,Cardiology,Heart Failure
5,What is the mechanism of action of sacubitril ...,"Sacubitril inhibits neprilysin, preventing deg...",Hard,Cardiology,Heart Failure
6,What is the target LDL-C level for patients at...,< 1.4 mmol/L (55 mg/dL) and a reduction of ≥ 5...,Medium,Cardiology,Dyslipidaemia
7,What is the first-line pharmacological treatme...,"High-intensity statin therapy (e.g., atorvasta...",Easy,Cardiology,Dyslipidaemia
8,What is the mechanism of action of PCSK9 inhib...,"PCSK9 inhibitors (e.g., evolocumab, alirocumab...",Hard,Cardiology,Dyslipidaemia
9,"In the context of acute coronary syndrome, wha...",The risk of death and ischemic events in patie...,Hard,Cardiology,Acute Coronary Syndrome


## 2. Loading the Base Model
We load `google/gemma-3-1b-it` using the standard Hugging Face `transformers` library. This version of Gemma is already instruction-tuned, making it a strong baseline for general Q&A.

In [3]:
MODEL_NAME = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
    device_map=device,
    use_cache=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## 3. Baseline Evaluation Loop
In this step, we loop through our 25 cardiology questions and perform two main tasks:
1. **Generation**: Ask the model the question and capture its response.
2. **Perplexity**: Measure how 'surprised' the model is by the correct (reference) answer. Lower perplexity indicates the model's internal knowledge aligns better with the target domain.

In [4]:
results_list = []
instructions = "\nProvide a concise medical answer based on clinical guidelines."

model.eval()
for index, row in tqdm(data.iterrows(), total=len(data)):
    question = row["Question"]
    answer = row["Answer"]
    difficulty = row["Difficulty"]

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": question + instructions}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
        generated_token_ids = outputs[0, inputs.shape[-1] :]
        generated_text = tokenizer.decode(
            generated_token_ids, skip_special_tokens=True
        ).strip()

        # Perplexity calculation
        reference_ids = tokenizer(
            answer, return_tensors="pt", add_special_tokens=False
        ).input_ids.to(device)
        full_input_ids = torch.cat([inputs, reference_ids], dim=1)
        labels = full_input_ids.clone()
        labels[:, : inputs.shape[1]] = -100
        loss = model(full_input_ids, labels=labels).loss
        perplexity = torch.exp(loss).item()

    results_list.append(
        {
            "question": question,
            "expected_answer": answer,
            "generated_answer": generated_text,
            "difficulty": difficulty,
            "perplexity": perplexity,
        }
    )

results_df = pd.DataFrame(results_list)
results_df.head()

  0%|          | 0/25 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
100%|██████████| 25/25 [01:13<00:00,  2.96s/it]


,question,expected_answer,generated_answer,difficulty,perplexity
0,What is the recommended first-line treatment f...,"ACE inhibitors or ARBs, combined with a calciu...",According to the European Society of Cardiolog...,Easy,103.225327
1,What is the definition of Stage 1 hypertension...,Systolic blood pressure 130–139 mmHg or diasto...,"According to the 2017 ACC/AHA guidelines, **St...",Medium,15.903885
2,What is the definition of hypertensive emergen...,Hypertensive emergency is severe hypertension ...,"Okay, here’s a concise medical answer regardin...",Hard,76.644775
3,In patients with heart failure with reduced ej...,"ACEi/ARNI, beta-blockers, MRA (mineralocortico...","In patients with HFrEF, the four pillars of me...",Easy,36.721043
4,What is the ejection fraction threshold that d...,"LVEF ≥ 50%, in the presence of relevant struct...","According to the ESC 2021 guidelines, the ejec...",Medium,738.002869


## 4. Scoring Metrics Implementation
To quantify the model's performance, we implement three scoring metrics:

### A. Keyword Matching (Accuracy Score)
We extract key medical terms from the expected answer and check their presence in the generated response. This is a basic but effective way to ensure 'hard facts' are present.

In [5]:
def calculate_keyword_score(generated, expected):
    # Simple keyword extraction: words longer than 4 chars, normalized
    expected_keywords = set(re.findall(r"\b\w{5,}\b", expected.lower()))
    if not expected_keywords:
        return 1.0

    generated_lower = generated.lower()
    found = sum(1 for kw in expected_keywords if kw in generated_lower)
    return found / len(expected_keywords)

### B. Semantic Similarity (Sentence-BERT)
Keyword matching fails if the model uses synonyms (e.g., 'HFrEF' vs 'Heart failure with reduced ejection fraction'). We use a pre-trained Sentence-Transformer (`all-MiniLM-L6-v2`) to measure the cosine similarity between the embeddings of the generated and expected answers.

In [6]:
similarity_model = SentenceTransformer("all-MiniLM-L6-v2")


def calculate_semantic_similarity(generated, expected):
    emb1 = similarity_model.encode(generated, convert_to_tensor=True)
    emb2 = similarity_model.encode(expected, convert_to_tensor=True)
    return util.pytorch_cos_sim(emb1, emb2).item()

### C. AI Judge (LLM-as-a-Judge)
Finally, we use a high-capacity model (**Qwen 3 7B Instruct**) to grade the responses. Using a larger, state-of-the-art model as a judge provides more reliable and nuanced evaluation than self-grading. We load it in 4-bit quantization to fit within memory constraints.

In [7]:
JUDGE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading AI Judge: {JUDGE_MODEL_NAME}...")

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)


def ai_judge_score(question, generated, expected):
    judge_prompt = f"""You are a medical board examiner. 
Rate the following generated answer on a scale from 1 to 10 based on its accuracy compared to the reference answer.

Question: {question}
Reference Answer: {expected}
Generated Answer: {generated}

Rules:
- 10: Perfectly accurate and comprehensive.
- 7-9: Mostly accurate with minor omissions.
- 4-6: Partially correct but missing key clinical details.
- 1-3: Incorrect or dangerously misleading.

Return ONLY the numeric score (e.g., '8')."""

    messages = [{"role": "user", "content": judge_prompt}]
    inputs = judge_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(judge_model.device)

    with torch.no_grad():
        outputs = judge_model.generate(inputs, max_new_tokens=10, do_sample=False)
        score_text = judge_tokenizer.decode(
            outputs[0, inputs.shape[-1] :], skip_special_tokens=True
        ).strip()

    match = re.search(r"\d+", score_text)
    if match:
        return int(match.group(0)) / 10.0
    return 0.5

Loading AI Judge: Qwen/Qwen2.5-7B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## 5. Running the Scoring Pipeline
Now we apply all three metrics to our results and combine them into a final **weighted score**.

In [8]:
print("Scoring baseline results...")
tqdm.pandas()

results_df["keyword_score"] = results_df.progress_apply(
    lambda x: calculate_keyword_score(x["generated_answer"], x["expected_answer"]),
    axis=1,
)

results_df["semantic_score"] = results_df.progress_apply(
    lambda x: calculate_semantic_similarity(
        x["generated_answer"], x["expected_answer"]
    ),
    axis=1,
)

results_df["ai_judge_score"] = results_df.progress_apply(
    lambda x: ai_judge_score(
        x["question"], x["generated_answer"], x["expected_answer"]
    ),
    axis=1,
)

# Final Score: weighted average (40% AI Judge, 40% Semantic, 20% Keywords)
results_df["final_score"] = (
    results_df["keyword_score"] * 0.2
    + results_df["semantic_score"] * 0.4
    + results_df["ai_judge_score"] * 0.4
)

results_df.to_csv("medical_baseline_results.csv", index=False)
results_df[["question", "perplexity", "final_score"]].head()

Scoring baseline results...


100%|██████████| 25/25 [00:08<00:00,  2.83it/s]


,question,perplexity,final_score
0,What is the recommended first-line treatment f...,103.225327,0.374516
1,What is the definition of Stage 1 hypertension...,15.903885,0.802997
2,What is the definition of hypertensive emergen...,76.644775,0.680155
3,In patients with heart failure with reduced ej...,36.721043,0.838141
4,What is the ejection fraction threshold that d...,738.002869,0.361015


## 6. Results Visualization and Analysis
Finally, we aggregate the scores by difficulty. This helps us identify if the model struggles more with complex clinical scenarios compared to basic terminology.

In [9]:
summary = (
    results_df.groupby("difficulty")
    .agg(
        {
            "perplexity": "mean",
            "keyword_score": "mean",
            "semantic_score": "mean",
            "ai_judge_score": "mean",
            "final_score": "mean",
        }
    )
    .round(3)
)

print("--- Baseline Performance Summary ---")
display(summary)

print(f"\nOverall Baseline Mean Final Score: {results_df['final_score'].mean():.3f}")
print(f"Overall Baseline Mean Perplexity: {results_df['perplexity'].mean():.3f}")

--- Baseline Performance Summary ---


,perplexity,keyword_score,semantic_score,ai_judge_score,final_score
difficulty,,,,,
Easy,672.393,0.395,0.486,0.600,0.513
Hard,714.964,0.349,0.609,0.675,0.584
Medium,737.330,0.351,0.525,0.533,0.493



Overall Baseline Mean Final Score: 0.529
Overall Baseline Mean Perplexity: 709.393


### Conclusion
The scores above represent the **untrained capability** of the model in the cardiology domain. 

**Observations to look for:**
- High perplexity on 'Hard' questions indicates the model finds clinical guidelines less predictable.
- Lower AI Judge scores on specific sub-topics highlight domain-specific gaps.

In the next notebooks, we will use **Tunix** to fine-tune this model on medical data and observe how these metrics improve!